# V4 Universal Football Model - Footballdata.io Ingestion

This notebook tests the Footballdata.io API and prepares data for the V4 model.

In [17]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")
if not API_KEY:
    print("API key not found. Please check ../.env.")
else:
    print(f"API key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

API key loaded: fd_75...6542e


## 1. Basic API Fetch Function

In [18]:
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Fetch a JSON payload from Footballdata.io."""
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Dynamic Fixture Parsing
The Footballdata `/fixtures/today` endpoint returns `data` as a dictionary (grouped by league or date) rather than a flat list. We will safely unpack it.

In [19]:
data = fetch_footballdata("leagues")
print(json.dumps(data, indent=2)[:500] + "\n...[truncated]")

{
  "success": true,
  "data": [
    {
      "league_id": 15,
      "league_name": "Premier League",
      "country": "England",
      "league_image": "https://footballdata.io/img/league/england-premier-league.png",
      "seasons_available": 20,
      "earliest_season": 20072008,
      "latest_season": 20262027
    },
    {
      "league_id": 45,
      "league_name": "UEFA Champions League",
      "country": "Europe",
      "league_image": "https://footballdata.io/img/league/europe-uefa-champio
...[truncated]


## 3. Inspecting Match Endpoints
Inspect the response shape before selecting a match ID, since `data` may be a list or dictionary.

In [20]:
print("Fetching /fixtures/today...")
today_data = fetch_footballdata("fixtures/today")
match_id_to_test = None

if today_data and today_data.get("success") and "data" in today_data:
    data_payload = today_data["data"]
    
    if isinstance(data_payload, list) and len(data_payload) > 0:
        match_id_to_test = data_payload[0].get("match_id")
    elif isinstance(data_payload, dict):
        # Iterate through the keys (which might be league IDs or dates) to find the first match array
        for key, val in data_payload.items():
            if isinstance(val, list) and len(val) > 0:
                match_id_to_test = val[0].get("match_id")
                break
            elif isinstance(val, dict) and "matches" in val:
                match_list = val["matches"]
                if len(match_list) > 0:
                    match_id_to_test = match_list[0].get("match_id")
                    break

if match_id_to_test:
    print(f"\n✅ Found matches today! We will use match_id: {match_id_to_test} for detailed tests.")
else:
    print("\n⚠️ Could not automatically extract a match_id. Using the fallback match provided.")
    match_id_to_test = 780100645 # Manually testing the match you provided earlier

Fetching /fixtures/today...

✅ Found matches today! We will use match_id: 780100645 for detailed tests.


## 3. Deep Dive into Stats & Events
Let's fetch the granular match data so we can map `live_xg` and `red_cards`.

In [21]:
if match_id_to_test:
    print(f"Fetching stats for match {match_id_to_test}...")
    stats_data = fetch_footballdata(f"matches/{match_id_to_test}/stats")
    
    if stats_data and stats_data.get("success"):
        print("\n📊 STATS DATA KEYS:", list(stats_data["data"].keys()))
        if "statistics" in stats_data["data"]:
            print("\nSnippet of 'statistics':")
            print(json.dumps(stats_data["data"]["statistics"], indent=2)[:800])
        else:
            print("\nSnippet of full stats payload:")
            print(json.dumps(stats_data["data"], indent=2)[:800])
            
    print("\n" + "="*50 + "\n")
    
    print(f"Fetching events for match {match_id_to_test}...")
    events_data = fetch_footballdata(f"matches/{match_id_to_test}/events")
    
    if events_data and events_data.get("success"):
        print("\n⏱️ EVENTS DATA KEYS:", list(events_data["data"].keys()))
        if "events" in events_data["data"]:
            print("\nSnippet of 'events':")
            print(json.dumps(events_data["data"]["events"], indent=2)[:800])
        else:
            print("\nSnippet of full events payload:")
            print(json.dumps(events_data["data"], indent=2)[:800])

Fetching stats for match 780100645...

📊 STATS DATA KEYS: ['match', 'stats', 'venue']

Snippet of full stats payload:
{
  "match": {
    "match_id": 780100645,
    "match_date": "2026-09-05 11:30:00",
    "date_unix": 1788607800,
    "status": "complete",
    "status_localized": "Finished",
    "league": {
      "league_id": 15,
      "name": "England Premier League",
      "country": "England",
      "competition_name": "Premier League",
      "image": "https://footballdata.io/img/league/england-premier-league.png"
    },
    "season": {
      "season_id": 103535,
      "year": 20262027
    },
    "home_team": {
      "team_id": 141,
      "team_name": "Newcastle United",
      "team_logo": "https://footballdata.io/img/team/newcastle-united-fc.png"
    },
    "away_team": {
      "team_id": 132,
      "team_name": "AFC Bournemouth",
      "team_logo": "https://footballdata.io/img/team/afc-bournemouth.png


Fetching events for match 780100645...

⏱️ EVENTS DATA KEYS: ['match', 'events'